# Stage 2 — Market microstructure fundamentals (real BTCUSDT_PERP data)

**Day:** 2026-04-06 (Monday, training period, median trading activity).
Everything below is computed from your real files, read straight out of the `.tar` archives.

Each section follows: **concept → intuition → maths → real example → code → what the data says**.
Numbers in the "what the data says" cells are measured on **one training day only**: they are
*exploratory*, not model results, and must not be used to choose features (that is Stage 9).

In [1]:
import sys, os
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))
import numpy as np, polars as pl, pyarrow.parquet as pq
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
from hft500 import micro as m
from hft500.tarsource import open_any
pl.Config.set_tbl_width_chars(160); pl.Config.set_tbl_cols(14); pl.Config.set_float_precision(4)

ROOT = Path.cwd().parent.parent            # .../short-horizon-challenge
DAY = "2026-04-06"
ARCH = {"BTCUSDT_PERP": "data-BTCUSDT_PERP-014.tar", "ETHUSDT_PERP": "data-ETHUSDT_PERP-003.tar",
        "SOLUSDT_PERP": "data-SOLUSDT_PERP-002.tar", "1000PEPEUSDT_PERP": "data-1000PEPEUSDT_PERP-008.tar"}

def load(table, inst="BTCUSDT_PERP", day=DAY):
    with open_any(f"{ROOT / ARCH[inst]}::data/{table}/{inst}/{day}.parquet") as fh:
        return pl.from_arrow(pq.read_table(fh))

meta = pl.read_parquet(ROOT / "data/meta/daily.parquet").filter(pl.col("date") == DAY)
TICK = meta.filter(pl.col("instrument") == "BTCUSDT_PERP")["tick_size"].item()
book, trades, tgt = load("book"), load("trades"), load("targets")
print(f"tick = {TICK:.2f} USDT | book rows {book.height:,} | trade bars {trades.height:,} | targets {tgt.height:,}")

tick = 0.10 USDT | book rows 172,795 | trade bars 372,449 | targets 172,800


## 1. Limit orders, market orders, bid, ask

* A **limit order** says "I will buy (or sell) up to *q* units at price *p* or better". If it cannot
  execute immediately it **rests** in the order book and provides **liquidity**. Whoever posts it is a
  **maker** (market makers do this continuously, on both sides).
* A **market order** (or an aggressive limit order) executes **immediately** against resting orders.
  Whoever sends it is a **taker**: it *consumes* liquidity and is the **aggressor** of the trade.
* **Best bid** = highest resting buy price; **best ask** = lowest resting sell price.
  `bid_qty` / `ask_qty` = size resting at exactly those prices (the **queues**), in BTC.

$$\text{mid} = \frac{\text{bid}+\text{ask}}{2}, \qquad \text{spread} = \text{ask}-\text{bid}, \qquad
\text{spread}_{bps} = \frac{\text{ask}-\text{bid}}{\text{mid}}\times 10^4$$

**Real row, 03:00:00.000 UTC.** Work it out by hand first, then check the cell below:
bid 69,183.6 (1.371 BTC) · ask 69,183.7 (6.428 BTC).
mid = 69,183.65 · spread = 0.1 USDT = **1 tick** = 0.1 / 69,183.65 × 10⁴ ≈ **0.0145 bps**.

In [2]:
T1 = 1775444400000   # 2026-04-06 03:00:00.000 UTC in epoch-ms
row = book.filter(pl.col("ts") == T1)
row.select("bid_px", "ask_px", "bid_qty", "ask_qty",
           m.mid().alias("mid"), m.spread().alias("spread"),
           m.spread_ticks(TICK).alias("spread_ticks"), m.spread_bps().alias("spread_bps"))

bid_px,ask_px,bid_qty,ask_qty,mid,spread,spread_ticks,spread_bps
f64,f64,f32,f32,f64,f64,f64,f64
69183.6000,69183.7000,1.3710,6.4280,69183.6500,0.1000,1.0000,0.0145


## 2. Tick size and the one-tick spread

The **tick** is the smallest price step. When the queues at the best prices are deep relative to how fast
orders arrive, the spread sits at exactly **1 tick** almost all the time. Then the *price* barely carries
information (it is pinned), and the information is in the **sizes**. That is why queue imbalance can
predict the next tick.

Below: for all four targets on the same day, one tick as a fraction of price (`tick_bps`) and the share
of 500 ms snapshots with a 1-tick spread.

In [3]:
rows = []
for inst in ARCH:
    b = load("book", inst)
    tick = meta.filter(pl.col("instrument") == inst)["tick_size"].item()
    s = b.select(m.spread_ticks(tick).alias("t"), m.spread_bps().alias("bps"), m.mid().alias("mid"))
    rows.append(dict(instrument=inst, tick=tick, mid_median=s["mid"].median(),
                     tick_bps=tick / s["mid"].median() * 1e4,
                     pct_1tick=(s["t"] == 1).mean() * 100, spread_bps_median=s["bps"].median()))
spreads = pl.DataFrame(rows); spreads

instrument,tick,mid_median,tick_bps,pct_1tick,spread_bps_median
str,f64,f64,f64,f64,f64
"""BTCUSDT_PERP""",0.1000,69373.6500,0.0144,99.8663,0.0144
"""ETHUSDT_PERP""",0.0100,2138.0850,0.0468,99.9259,0.0468
"""SOLUSDT_PERP""",0.0100,81.9950,1.2196,99.9931,1.2196
"""1000PEPEUSDT_PERP""",0.0000,0.0035,0.2854,99.6111,0.2854


**What the data says (this day):** all four targets have a 1-tick spread in **≥ 99.6%** of snapshots,
but their ticks differ enormously relative to price. SOL's tick is **1.22 bps**, about 85× BTC's
**0.014 bps** (ETH 0.047, PEPE 0.29). So when SOL's mid moves, it jumps by at least 0.6 bps; BTC's mid
can move in tiny steps.

Generalisation: a 500 ms SOL target is mostly 0 or ± half a tick (±0.61 bps), which makes it very
"discrete". BTC returns are smoother. Expect imbalance to behave differently per instrument, and check all
four before assuming one model fits all.

## 3. Queue imbalance and microprice

$$\text{imb} = \frac{Q_b - Q_a}{Q_b + Q_a}\in[-1,1], \qquad
\text{microprice} = \frac{\text{bid}\cdot Q_a + \text{ask}\cdot Q_b}{Q_b+Q_a}
= \text{mid} + \text{imb}\cdot\frac{\text{spread}}{2}$$

**Intuition.** If the ask queue is thin, a few market buys will empty it and the ask moves up;
the bid queue is thick, so it is unlikely to be emptied first. So the *next* mid move is more likely
**toward the thin side**: imb < 0 (more size on the ask) leans **down**.

**By hand, 03:00:00.000:** imb = (1.371 − 6.428)/(1.371 + 6.428) = −5.057/7.799 = **−0.6484**.
microprice = 69,183.65 + (−0.6484)(0.1/2) = **69,183.6176**, i.e. 0.32 tick below mid.

**What happened next (real):** at 03:00:00.500 the book was 69,180.0 / 69,180.1, so the mid fell
3.6 USDT. The supplied target y_500ms(03:00:00.000) = ln(69,180.05/69,183.65) = **−0.52 bps**. The
imbalance pointed the right way. One row is an anecdote though; the next cell measures the whole day.

In [4]:
day = (book.select("ts", m.queue_imbalance().alias("imb"),
                   m.microprice_dev_ticks(TICK).alias("mp_dev_ticks"),
                   m.depth_imbalance(1).alias("dimb_1"), m.depth_imbalance(4).alias("dimb_4"),
                   m.depth_imbalance(11).alias("dimb_11"))
          .join(tgt, on="ts", how="inner")
          .with_columns([(pl.col(h) * 1e4).alias(h + "_bps") for h in ("y_500ms", "y_2s", "y_8s")]))
print(f"rows: {day.height:,}   share of y_500ms exactly 0: {(day['y_500ms'] == 0).mean():.1%}")
dec = (day.with_columns(((pl.col("imb").rank("ordinal") - 1) * 10 // pl.len() + 1).alias("decile"))
          .group_by("decile").agg(pl.col("imb").mean().alias("imb_mean"),
                                  pl.col("y_500ms_bps").mean().alias("next_500ms_bps"),
                                  pl.col("y_2s_bps").mean().alias("next_2s_bps"),
                                  pl.len().alias("n"))
          .sort("decile"))
dec

rows: 172,795   share of y_500ms exactly 0: 80.4%


decile,imb_mean,next_500ms_bps,next_2s_bps,n
u32,f64,f64,f64,u32
1,-0.9357,-0.2331,-0.4640,17280
2,-0.7321,-0.0893,-0.2261,17279
3,-0.5498,-0.0457,-0.1282,17280
4,-0.3509,-0.0265,-0.0831,17279
5,-0.1206,-0.0050,-0.0012,17280
6,0.1184,0.0049,0.0187,17279
7,0.3505,0.0291,0.0736,17280
8,0.5492,0.0515,0.1448,17279
9,0.7356,0.0812,0.2210,17280


In [5]:
fig, ax = plt.subplots(figsize=(7, 3.6))
x = dec["imb_mean"].to_numpy()
ax.plot(x, dec["next_500ms_bps"], "o-", label="next 500 ms")
ax.plot(x, dec["next_2s_bps"], "s-", label="next 2 s")
ax.axhline(0, color="grey", lw=0.8); ax.axvline(0, color="grey", lw=0.8)
ax.set_xlabel("queue imbalance (decile mean)"); ax.set_ylabel("mean future return (bps)")
ax.set_title(f"BTCUSDT_PERP {DAY}: imbalance vs next return (1 day, exploratory)")
ax.legend(frameon=False); fig.tight_layout(); fig.savefig("stage2_imbalance_deciles.png", dpi=130)

## 4. Depth buckets

`bid_vKK` is the **notional (USDT)** resting on the bid between bucket edges, measured in bps from mid:
bucket 1 = 0–0.5 bps, 2 = 0.5–1, 3 = 1–1.5, 4 = 1.5–2, 5 = 2–10, … 11 = 75–100 bps.
Because notional is in USDT, it is comparable across price levels and (after normalising) across coins.

`depth_imbalance(k)` uses the **cumulative** notional of buckets 1..k: k = 1 is the touch, k = 4 is
"within 2 bps", k = 11 is "within 1%". Deeper liquidity changes slowly and tends to say more about
longer horizons than the touch does. That is a **hypothesis**; the next cell is one day of evidence.

In [6]:
cols = ["imb", "mp_dev_ticks", "dimb_1", "dimb_4", "dimb_11"]
corr = pl.DataFrame([{"feature": c, **{h: float(np.corrcoef(day[c], day[h])[0, 1]) for h in ("y_500ms", "y_2s", "y_8s")}}
                     for c in cols])
corr   # Pearson correlation with the (unclipped) target, this ONE day

feature,y_500ms,y_2s,y_8s
str,f64,f64,f64
"""imb""",0.2262,0.2242,0.1555
"""mp_dev_ticks""",0.1549,0.1370,0.0905
"""dimb_1""",0.2267,0.2231,0.1505
"""dimb_4""",0.1925,0.1854,0.1213
"""dimb_11""",0.0008,0.0015,0.0006


**What the depth table says (this day, exploratory):**
* `imb` (best-level queues, in BTC) and `dimb_1` (notional within 0.5 bps) are nearly identical, both
  ≈ 0.23 correlation with the next 500 ms return. The touch carries the signal.
* `dimb_4` (within 2 bps) is weaker (≈ 0.19); `dimb_11` (within 1%) is **≈ 0**. Deep liquidity says
  nothing at these horizons, at least on this day.
* `mp_dev_ticks` is weaker than `imb` even though microprice − mid = imb × spread/2. The reason is the
  rare wide-spread snapshots: they multiply the value and create outliers, and **Pearson is sensitive to
  outliers**. Same information, different scaling, different score. Remember this for Stage 7.

## 5. Trades and aggressor flow

Each 100 ms bar labelled `ts` covers trades in **(ts − 100, ts]**. `buy_*` = buyer-initiated (a taker
**lifted the ask**), `sell_*` = seller-initiated (a taker **hit the bid**).

$$\text{signed volume} = \text{buy\_qty} - \text{sell\_qty}, \qquad
\text{trade imbalance} = \frac{\text{buy\_qty}-\text{sell\_qty}}{\text{buy\_qty}+\text{sell\_qty}}$$

**Order flow vs resting liquidity.** The book shows *intentions* (resting orders, which can be cancelled
without trading); trades show *actions* (someone paid the spread to trade now). A burst of
seller-initiated volume eats the bid queues and pushes the mid down.

**Real example, 14:00:00 UTC:** in the bar (14:00:00.000, 14:00:00.100] takers **sold 11.514 BTC in 501
trades** and bought nothing. Over the next 400 ms the best bid fell from 69,399.7 to 69,349.9 (−50 USDT,
about −7 bps). Note the pattern: a burst at the top of the hour.

Causality reminder: to predict at time t you may only use bars with ts ≤ t. Below, each grid point t
gets the sum of the 5 bars in (t − 500, t], all already finished at t.

In [7]:
flow = (trades.with_columns(((pl.col("ts") + 499) // 500 * 500).alias("t"))   # bar end -> the grid point it is known at
              .group_by("t").agg(pl.col("buy_qty").sum(), pl.col("sell_qty").sum())
              .rename({"t": "ts"}))
d2 = (day.join(flow, on="ts", how="left").fill_null(0.0)
         .with_columns(m.signed_volume().alias("sv_500ms"), m.trade_imbalance().alias("ti_500ms")))
pl.DataFrame([{"feature": c, **{h: float(np.corrcoef(d2[c], d2[h])[0, 1]) for h in ("y_500ms", "y_2s", "y_8s")}}
              for c in ("sv_500ms", "ti_500ms")])

feature,y_500ms,y_2s,y_8s
str,f64,f64,f64
"""sv_500ms""",0.0478,0.0560,0.0246
"""ti_500ms""",0.0828,0.0672,0.0411


In [8]:
T14 = 1775484000000
w = (book.filter(pl.col("ts").is_between(T14 - 5000, T14 + 5000)).select("ts", m.mid().alias("mid"))
         .join(flow, on="ts", how="left").fill_null(0.0).with_columns(((pl.col("ts") - T14) / 1000).alias("sec")))
fig, ax = plt.subplots(2, 1, figsize=(7, 4.4), sharex=True)
ax[0].plot(w["sec"], w["mid"], "-o", ms=3); ax[0].set_ylabel("mid (USDT)")
ax[1].bar(w["sec"], w["buy_qty"], width=0.4, label="buy (taker)"); ax[1].bar(w["sec"], -w["sell_qty"], width=0.4, label="sell (taker)")
ax[1].set_ylabel("BTC per 500 ms"); ax[1].set_xlabel("seconds from 14:00:00 UTC"); ax[1].legend(frameon=False)
ax[0].set_title("Seller-initiated burst at the top of the hour"); fig.tight_layout(); fig.savefig("stage2_flow_1400.png", dpi=130)

## 6. Why can short-horizon moves be (a little) predictable?

1. **Mechanical queue depletion.** A mid move needs a queue to empty. Thin queues empty first, so
   imbalance predicts the *direction* of the next tick (Section 3).
2. **Order-flow persistence.** Large orders are split into many small ones, so buying tends to follow
   buying for a few seconds (Section 5).
3. **Price discovery across venues.** Information often reaches one market (e.g. BTC perp) before
   another (alts, spot). That is Stage 8.

**Statistical vs tradable.** A correlation of a few percent is valuable for this assignment's metric,
but the predicted moves are mostly smaller than the spread plus fees. "Predictable" here does **not**
mean "profitable".

**Two facts to keep from this day:** most 500 ms targets are exactly 0 (the mid did not move), and
the spread is almost always one tick. Both make the 500 ms target "spiky": many zeros, a few ±1-tick
jumps. That is why target clipping and Pearson's outlier sensitivity matter (Stage 5).

## 7. Exercises (answers at the bottom)

1. A book shows bid 2,000.00 (40 ETH), ask 2,000.01 (10 ETH). Compute mid, spread in bps, imbalance
   and microprice. Which way does the microprice lean?
2. Why is `bid_v01` measured in USDT rather than BTC? What breaks if you compare raw BTC sizes between
   BTC and PEPE?
3. A trade bar has `ts = 12:00:00.300`. At which prediction times t on the 500 ms grid may you use it?
4. In the decile table, is the relationship symmetric? Is it roughly linear? What does that suggest for
   a *linear* model like the baseline?
5. Why must `trade_imbalance` return 0 (not NaN) when no trades happened?

### Answers
1. mid = 2,000.005; spread = 0.01/2,000.005 × 10⁴ = 0.05 bps; imb = (40 − 10)/50 = +0.6;
   microprice = 2,000.005 + 0.6 × 0.005 = 2,000.008. It leans **up**, toward the thin ask.
2. Notional is comparable across instruments and price levels: 1 BTC ≈ 69,000 USDT but
   1,000 PEPE is a tiny amount. Raw sizes would make "big" mean different things per coin.
3. The bar covers (12:00:00.200, 12:00:00.300] and is known at 12:00:00.300, so it may be used for every
   t ≥ 12:00:00.300. The first grid point is 12:00:00.500. Using it at 12:00:00.000 would be look-ahead.
4. Look at your own table. Typically the extreme deciles carry most of the signal and the middle is flat.
   A linear model can't bend, so it under-uses the extremes; a later model (trees) can.
5. NaN propagates into every later calculation and the row is dropped or poisons a model. "No trades"
   is real information: zero pressure.